# Supplementary Tables -- BATTLE-AMP

Generates publication-ready LaTeX for supplementary tables:

- **Table S2**: Full classification metrics (21 models x 27 tasks)
- **Table S3**: Full regression metrics (14 regressors x 2 strain-level tasks)
- **Table S4**: Resource usage summary

**Classification metrics** (Section 2): MCC, FPR, AUROC, AUPRC,
pAUROC$_{0.1}$, pAUROC$_{0.01}$, Precision@$k$, LR$+$.

**Regression metrics** (Section 2): $R^2_{\log_2}$, Spearman $\rho$,
RMSL$^2$E. Predictions clamped to 0.25--512 $\mu$g/ml.

All numeric values rounded to 2 decimals; LR+ to 1 decimal.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

## 1. Configuration

In [7]:
RESULTS_DIR = Path('../results/aggregated')

EXCLUDE_MODELS = {'example-model', 'mole-amp', 'mole-amp-max'}

EXCLUDE_CLS_TASKS = {
    'example_classification',
    'high_similarity',
    'deepamp_classification',
    'deepamp_species_gramneg',
    'deepamp_species_grampos',
}

In [8]:
# ---- Model display names ----

MODEL_DISPLAY = {
    'ampeppy':                'amPEPpy',
    'amplify':                'AMPlify',
    'ampredictor':            'AMPredictor',
    'ampredmfa':              'AMPredMFA',
    'ampscanner':             'AMPScanner$_{\\mathrm{v2}}$',
    'apex-abaumannii':        'APEX$_{\\mathrm{Ab}}$',
    'apex-ecoli':             'APEX$_{\\mathrm{Ec}}$',
    'apex-kpneumoniae':       'APEX$_{\\mathrm{Kp}}$',
    'apex-min':               'APEX$_{\\mathrm{min}}$',
    'apex-paeruginosa':       'APEX$_{\\mathrm{Pa}}$',
    'apex-saureus':           'APEX$_{\\mathrm{Sa}}$',
    'deep-amp-cnn-gramneg':   'DeepAMP$_{\\mathrm{CNN, G-}}$',
    'deep-amp-cnn-grampos':   'DeepAMP$_{\\mathrm{CNN, G+}}$',
    'deep-amp-lstm-gramneg':  'DeepAMP$_{\\mathrm{LSTM, G-}}$',
    'deep-amp-lstm-grampos':  'DeepAMP$_{\\mathrm{LSTM, G+}}$',
    'hydramp-amp-classifier': 'HydrAMP$_{\\mathrm{AMP}}$',
    'hydramp-mic-classifier': 'HydrAMP$_{\\mathrm{Ec}}$',
    'mbc-attention':          'MBC-Attention',
    'sensexamp-classifier':   'SenseXAMP$_{\\mathrm{cls}}$',
    'sensexamp-ecoli':        'SenseXAMP$_{\\mathrm{Ec}}$',
    'sensexamp-saureus':      'SenseXAMP$_{\\mathrm{Sa}}$',
}

MODEL_ORDER = list(MODEL_DISPLAY.keys())

In [9]:
# ---- Classification task display names and grouping ----
# Groups follow Table 1 in the main text.

TASK_DISPLAY = {
    'amp':                     'AMP/non-AMP',
    'broad_activity':          'GeneralActivity',
    'gram_plus':               'Gram$+$',
    'gram_minus':              'Gram$-$',
    'species_saureus':         '\\textit{S.~aureus}',
    'species_ecoli':           '\\textit{E.~coli}',
    'species_abaumannii':      '\\textit{A.~baumannii}',
    'species_kpneumoniae':     '\\textit{K.~pneumoniae}',
    'species_paeruginosa':     '\\textit{P.~aeruginosa}',
    'strain_saureus25923':     '\\textit{S.~aureus} ATCC 25923',
    'strain_saureus33591':     '\\textit{S.~aureus} ATCC 33591',
    'strain_saureus43300':     '\\textit{S.~aureus} ATCC 43300',
    'strain_ecoli25922':       '\\textit{E.~coli} ATCC 25922',
    'strain_abaumannii19606':  '\\textit{A.~baumannii} ATCC 19606',
    'strain_kpneumoniae700603':'\\textit{K.~pneumoniae} ATCC 700603',
    'strain_paeruginosa27853': '\\textit{P.~aeruginosa} ATCC 27853',
    'length_01_10':            '1--10 aa',
    'length_11_20':            '11--20 aa',
    'length_21_30':            '21--30 aa',
    'length_31_50':            '31--50 aa',
    'homology_80':             '80\\% identity',
    'homology_60':             '60\\% identity',
    'homology_40':             '40\\% identity',
    'synthetic_shuffled':      'SyntheticShuffled',
    'synthetic_realistic':     'SyntheticRealistic',
    'synthetic_random':        'SyntheticRandom',
    'slay':                    'SLAY',
}

TASK_GROUPS = {
    'AMP/non-AMP':      ['amp'],
    'GeneralActivity':  ['broad_activity'],
    'GramActivity':     ['gram_plus', 'gram_minus'],
    'SpeciesActivity': [
        'species_saureus', 'species_ecoli', 'species_abaumannii',
        'species_kpneumoniae', 'species_paeruginosa',
    ],
    'StrainActivity': [
        'strain_saureus25923', 'strain_saureus33591',
        'strain_saureus43300', 'strain_ecoli25922',
        'strain_abaumannii19606', 'strain_kpneumoniae700603',
        'strain_paeruginosa27853',
    ],
    'LengthSplit':      [
        'length_01_10', 'length_11_20', 'length_21_30', 'length_31_50',
    ],
    'HomologySplit':    ['homology_80', 'homology_60', 'homology_40'],
    'Synthetic decoys': [
        'synthetic_shuffled', 'synthetic_realistic', 'synthetic_random',
    ],
    'SLAY':             ['slay'],
}

In [10]:
# ---- Regression: strain-level tasks only ----

REG_TASK_DISPLAY = {
    'regression_ecoli25922':   '\\textit{E.~coli} ATCC 25922',
    'regression_saureus25923': '\\textit{S.~aureus} ATCC 25923',
}

REG_TASK_ORDER = ['regression_ecoli25922', 'regression_saureus25923']

EXCLUDE_REG_TASKS = {
    'deepamp_regression_gramneg',
    'deepamp_regression_grampos',
}

In [11]:
# ---- Metrics ----

CLS_METRICS = [
    'coverage', 'mcc', 'fpr', 'auroc', 'auprc',
    'pauroc_01', 'pauroc_001', 'precision_at_k', 'lr_plus',
]

CLS_METRIC_LABELS = {
    'coverage':       'Cov.',
    'mcc':            'MCC',
    'fpr':            'FPR',
    'auroc':          'AUROC',
    'auprc':          'AUPRC',
    'pauroc_01':      'pAUROC$_{0.1}$',
    'pauroc_001':     'pAUROC$_{0.01}$',
    'precision_at_k': 'P@$k$',
    'lr_plus':        'LR$+$',
}

REG_METRICS = ['coverage', 'r2_log2', 'spearman', 'rmsl2e']

REG_METRIC_LABELS = {
    'coverage':  'Cov.',
    'r2_log2':   '$R^2_{\\log_2}$',
    'spearman':  '$\\rho_s$',
    'rmsl2e':    'RMSL$^2$E',
}

# True = higher is better
HIGHER_IS_BETTER = {
    'coverage': True,
    'mcc': True, 'fpr': False, 'auroc': True, 'auprc': True,
    'pauroc_01': True, 'pauroc_001': True,
    'precision_at_k': True, 'lr_plus': True,
    'r2_log2': True, 'spearman': True, 'rmsl2e': False,
}

# Per-metric decimal precision
DECIMALS = {m: 2 for m in CLS_METRICS + REG_METRICS}
DECIMALS['lr_plus'] = 1

print('Configuration loaded.')

Configuration loaded.


## 2. Load and clean data

In [12]:
cls_raw = pd.read_csv(RESULTS_DIR / 'classification_results.tsv', sep='\t')
reg_raw = pd.read_csv(RESULTS_DIR / 'regression_results.tsv', sep='\t')
res_summary = pd.read_csv(RESULTS_DIR / 'resource_summary.tsv', sep='\t')

print(f'Classification raw: {cls_raw.shape}')
print(f'Regression raw:     {reg_raw.shape}')
print(f'Resource summary:   {res_summary.shape}')

Classification raw: (704, 27)
Regression raw:     (64, 27)
Resource summary:   (13, 5)


In [13]:
# ---- Classification ----

cls = cls_raw[
    ~cls_raw['variant'].isin(EXCLUDE_MODELS)
    & ~cls_raw['task'].isin(EXCLUDE_CLS_TASKS)
].copy().reset_index(drop=True)

for col in ['coverage', 'mcc', 'fpr', 'tpr', 'tnr', 'auroc', 'auprc',
            'pauroc_01', 'pauroc_001', 'precision_at_k',
            'n_positive', 'n_negative']:
    cls[col] = pd.to_numeric(cls[col], errors='coerce')

# LR+ = TPR / FPR; NaN where FPR = 0
cls['lr_plus'] = np.where(
    cls['fpr'] == 0, np.nan, cls['tpr'] / cls['fpr'])

print(f'Classification: {cls.shape}')
print(f'  LR+ finite: [{cls["lr_plus"].min():.1f}, '
      f'{cls["lr_plus"].max():.1f}]')
print(f'  LR+ undefined (FPR=0): {cls["lr_plus"].isna().sum()}')

Classification: (567, 28)
  LR+ finite: [0.3, 975.6]
  LR+ undefined (FPR=0): 43


In [14]:
# ---- Regression: keep only strain-level tasks, drop ghosts ----

reg = reg_raw[
    ~reg_raw['variant'].isin(EXCLUDE_MODELS)
    & ~reg_raw['task'].isin(EXCLUDE_REG_TASKS)
].copy()

reg_metric_cols = ['r2_log2', 'spearman', 'msl2e', 'rmsl2e']
for col in reg_metric_cols + ['coverage']:
    reg[col] = pd.to_numeric(reg[col], errors='coerce')

ghost_mask = reg[reg_metric_cols].isna().all(axis=1)
n_ghost = ghost_mask.sum()
reg = reg[~ghost_mask].reset_index(drop=True)

print(f'Regression: {reg.shape} (dropped {n_ghost} ghost rows)')

Regression: (28, 27) (dropped 0 ghost rows)


## 3. Validation

In [15]:
cls_tasks = sorted(cls['task'].unique())
cls_models = sorted(cls['variant'].unique())
reg_tasks = sorted(reg['task'].unique())
reg_models = sorted(reg['variant'].unique())

print(f'Classification: {len(cls_models)} models x '
      f'{len(cls_tasks)} tasks = {len(cls)} rows '
      f'(expected {len(cls_models) * len(cls_tasks)})')
assert len(cls) == len(cls_models) * len(cls_tasks)

print(f'Regression:     {len(reg_models)} models x '
      f'{len(reg_tasks)} tasks = {len(reg)} rows '
      f'(expected {len(reg_models) * len(reg_tasks)})')
assert len(reg) == len(reg_models) * len(reg_tasks)

expected_cls = {t for g in TASK_GROUPS.values() for t in g}
assert expected_cls == set(cls_tasks), (
    f'Task mismatch: missing={expected_cls - set(cls_tasks)}, '
    f'extra={set(cls_tasks) - expected_cls}')

print('All checks passed.')

Classification: 21 models x 27 tasks = 567 rows (expected 567)
Regression:     14 models x 2 tasks = 28 rows (expected 28)
All checks passed.


## 4. LaTeX table utilities

In [16]:
def fmt_val(v, decimals=2, bold=False, is_best=False,
            is_lr_plus=False):
    """Format a single numeric value for LaTeX."""
    if pd.isna(v):
        return '$\\infty$' if is_lr_plus else '--'
    s = f'{v:.{decimals}f}'
    if bold and is_best:
        s = f'\\textbf{{{s}}}'
    return s


def find_best(series, higher_is_better=True):
    """Index of the best non-NaN value."""
    s = series.dropna()
    if s.empty:
        return None
    return s.idxmax() if higher_is_better else s.idxmin()

In [17]:
def build_cls_subtable(
    df, tasks, metrics, model_order, task_display,
    metric_labels, model_display, group_label,
    decimals_map=DECIMALS, bold=True,
):
    """One LaTeX tabular per task in a group."""
    sub = df[df['task'].isin(tasks)]
    lines = []

    for task in tasks:
        td = sub[sub['task'] == task].set_index('variant')
        task_label = task_display.get(task, task)
        n_m = len(metrics)

        best_idx = {}
        if bold:
            for m in metrics:
                if m == 'coverage':
                    continue
                best_idx[m] = find_best(
                    td[m], HIGHER_IS_BETTER.get(m, True))

        col_spec = 'l' + 'r' * n_m
        header = ' & '.join(
            metric_labels.get(m, m) for m in metrics)

        lines.append(f'% --- {group_label}: {task_label} ---')
        lines.append(f'\\begin{{tabular}}{{{col_spec}}}')
        lines.append('\\toprule')
        lines.append(
            f'\\multicolumn{{{n_m + 1}}}{{l}}'
            f'{{\\textbf{{{group_label}}}: {task_label}'
            f' ($n = {int(td["task_size"].iloc[0]):,}$)}} \\\\'
        )
        lines.append('\\midrule')
        lines.append(f'Model & {header} \\\\')
        lines.append('\\midrule')

        for variant in model_order:
            if variant not in td.index:
                continue
            row = td.loc[variant]
            name = model_display.get(variant, variant)
            vals = []
            for m in metrics:
                v = row.get(m, np.nan)
                d = decimals_map.get(m, 2)
                vals.append(fmt_val(
                    v, decimals=d,
                    bold=bold,
                    is_best=(best_idx.get(m) == variant),
                    is_lr_plus=(m == 'lr_plus'),
                ))
            lines.append(f'{name} & {" & ".join(vals)} \\\\')

        lines.append('\\bottomrule')
        lines.append('\\end{tabular}')
        lines.append('')

    return '\n'.join(lines)

In [18]:
def build_reg_table(
    df, tasks, metrics, model_order, task_display,
    metric_labels, model_display,
    decimals_map=DECIMALS, bold=True,
):
    """Single LaTeX tabular: rows = models, column groups = tasks."""
    n_m = len(metrics)
    n_t = len(tasks)
    reg_models = [m for m in model_order if m in df['variant'].values]

    best = {}
    if bold:
        for task in tasks:
            td = df[df['task'] == task].set_index('variant')
            for m in metrics:
                if m == 'coverage':
                    continue
                best[(task, m)] = find_best(
                    td[m], HIGHER_IS_BETTER.get(m, True))

    col_spec = 'l' + ('r' * n_m) * n_t
    lines = []
    lines.append(f'\\begin{{tabular}}{{{col_spec}}}')
    lines.append('\\toprule')

    # Task header
    h = ['']
    for task in tasks:
        h.append(f'\\multicolumn{{{n_m}}}{{c}}'
                 f'{{{task_display.get(task, task)}}}')
    lines.append(' & '.join(h) + ' \\\\')

    rules = []
    for i in range(n_t):
        s = 2 + i * n_m
        rules.append(f'\\cmidrule(lr){{{s}-{s + n_m - 1}}}')
    lines.append(' '.join(rules))

    # Metric subheader
    sh = ['Model']
    for _ in tasks:
        for m in metrics:
            sh.append(metric_labels.get(m, m))
    lines.append(' & '.join(sh) + ' \\\\')
    lines.append('\\midrule')

    for variant in reg_models:
        name = model_display.get(variant, variant)
        parts = [name]
        for task in tasks:
            row = df[(df['task'] == task)
                     & (df['variant'] == variant)]
            if row.empty:
                parts.extend(['--'] * n_m)
            else:
                r = row.iloc[0]
                for m in metrics:
                    v = r.get(m, np.nan)
                    d = decimals_map.get(m, 2)
                    parts.append(fmt_val(
                        v, decimals=d,
                        bold=bold,
                        is_best=(best.get((task, m)) == variant),
                    ))
        lines.append(' & '.join(parts) + ' \\\\')

    lines.append('\\bottomrule')
    lines.append('\\end{tabular}')
    return '\n'.join(lines)

## 5. Table S2: Full classification metrics

In [19]:
s2_parts = []

for group_name, group_tasks in TASK_GROUPS.items():
    valid = [t for t in group_tasks if t in cls['task'].values]
    if not valid:
        continue
    s2_parts.append(build_cls_subtable(
        cls, valid, CLS_METRICS, MODEL_ORDER,
        TASK_DISPLAY, CLS_METRIC_LABELS, MODEL_DISPLAY,
        group_label=group_name,
    ))

table_s2_body = '\n\n'.join(s2_parts)
print(f'Table S2 body: {len(table_s2_body):,} characters, '
      f'{table_s2_body.count(chr(10))} lines')

Table S2 body: 59,378 characters, 844 lines


In [20]:
# Preview: first two task groups
preview = build_cls_subtable(
    cls, ['amp', 'broad_activity'], CLS_METRICS, MODEL_ORDER,
    TASK_DISPLAY, CLS_METRIC_LABELS, MODEL_DISPLAY,
    group_label='Activity',
)
print(preview)

% --- Activity: AMP/non-AMP ---
\begin{tabular}{lrrrrrrrrr}
\toprule
\multicolumn{10}{l}{\textbf{Activity}: AMP/non-AMP ($n = 40,494$)} \\
\midrule
Model & Cov. & MCC & FPR & AUROC & AUPRC & pAUROC$_{0.1}$ & pAUROC$_{0.01}$ & P@$k$ & LR$+$ \\
\midrule
amPEPpy & 1.00 & 0.23 & 0.47 & 0.66 & 0.70 & 0.59 & 0.56 & \textbf{1.00} & 1.5 \\
AMPlify & 1.00 & 0.41 & 0.25 & 0.75 & 0.81 & 0.72 & 0.65 & 1.00 & 2.7 \\
AMPredictor & 0.89 & 0.34 & 0.49 & 0.72 & 0.73 & 0.55 & 0.50 & 0.60 & 1.6 \\
AMPredMFA & 1.00 & 0.09 & 0.80 & 0.59 & 0.61 & 0.55 & 0.51 & 1.00 & 1.1 \\
AMPScanner$_{\mathrm{v2}}$ & 0.85 & 0.65 & 0.08 & 0.86 & 0.89 & 0.81 & 0.70 & 1.00 & 9.3 \\
APEX$_{\mathrm{Ab}}$ & 0.81 & 0.20 & 0.00 & 0.54 & 0.54 & 0.54 & 0.54 & 1.00 & 107.9 \\
APEX$_{\mathrm{Ec}}$ & 0.81 & 0.16 & 0.00 & 0.52 & 0.50 & 0.52 & 0.52 & 1.00 & 167.5 \\
APEX$_{\mathrm{Kp}}$ & 0.81 & 0.03 & 0.00 & 0.53 & 0.51 & 0.50 & 0.50 & 0.60 & 13.8 \\
APEX$_{\mathrm{min}}$ & 0.81 & 0.38 & 0.01 & 0.64 & 0.71 & 0.63 & 0.60 & 1.00 & 27.0 \

## 6. Table S3: Full regression metrics

In [21]:
valid_reg_tasks = [
    t for t in REG_TASK_ORDER if t in reg['task'].values]

table_s3_body = build_reg_table(
    reg, valid_reg_tasks, REG_METRICS, MODEL_ORDER,
    REG_TASK_DISPLAY, REG_METRIC_LABELS, MODEL_DISPLAY,
)

print(table_s3_body)

\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{4}{c}{\textit{E.~coli} ATCC 25922} & \multicolumn{4}{c}{\textit{S.~aureus} ATCC 25923} \\
\cmidrule(lr){2-5} \cmidrule(lr){6-9}
Model & Cov. & $R^2_{\log_2}$ & $\rho_s$ & RMSL$^2$E & Cov. & $R^2_{\log_2}$ & $\rho_s$ & RMSL$^2$E \\
\midrule
AMPredictor & 1.00 & 0.23 & 0.52 & 1.75 & 1.00 & -0.02 & 0.32 & 2.22 \\
APEX$_{\mathrm{Ab}}$ & 1.00 & -0.73 & 0.26 & 2.61 & 1.00 & -0.37 & 0.31 & 2.56 \\
APEX$_{\mathrm{Ec}}$ & 1.00 & -0.91 & 0.25 & 2.74 & 1.00 & -0.44 & 0.33 & 2.62 \\
APEX$_{\mathrm{Kp}}$ & 1.00 & -1.83 & 0.04 & 3.33 & 1.00 & -1.17 & 0.18 & 3.22 \\
APEX$_{\mathrm{min}}$ & 1.00 & -0.34 & 0.23 & 2.29 & 1.00 & -0.06 & 0.34 & 2.25 \\
APEX$_{\mathrm{Pa}}$ & 1.00 & -1.35 & 0.23 & 3.04 & 1.00 & -0.85 & 0.29 & 2.97 \\
APEX$_{\mathrm{Sa}}$ & 1.00 & -1.76 & 0.08 & 3.29 & 1.00 & -1.00 & 0.30 & 3.09 \\
DeepAMP$_{\mathrm{CNN, G-}}$ & 1.00 & -0.12 & 0.32 & 2.10 & 1.00 & \textbf{0.12} & \textbf{0.53} & \textbf{2.05} \\
DeepAMP$_{\mathrm{CNN, G+}}

## 7. Table S4: Resource usage

In [22]:
res = res_summary[
    ~res_summary['variant'].isin(EXCLUDE_MODELS)
].sort_values('total_wall_s', ascending=False
).reset_index(drop=True)

RESOURCE_DISPLAY = {
    'ampeppy':                'AMPEPPy',
    'amplify':                'AMPlify',
    'ampredictor':            'AMPredictor',
    'ampredmfa':              'AMPredMFA',
    'ampscanner':             'AMPScanner',
    'apex':                   'APEX',
    'deep-amp':               'DeepAMP',
    'hydramp-amp-classifier': 'HydrAMP$_{\\mathrm{AMP}}$',
    'hydramp-mic-classifier': 'HydrAMP$_{\\mathrm{MIC}}$',
    'mbc-attention':          'MBC-Attention',
    'sensexamp':              'SenseXAMP',
}


def fmt_time(s):
    if pd.isna(s): return '--'
    if s < 60:   return f'{s:.0f}\\,s'
    if s < 3600: return f'{s / 60:.1f}\\,min'
    return f'{s / 3600:.1f}\\,h'


def fmt_mem(mb):
    if pd.isna(mb): return '--'
    if mb < 1024: return f'{mb:.0f}\\,MB'
    return f'{mb / 1024:.1f}\\,GB'


s4 = [
    '\\begin{tabular}{lrrr}',
    '\\toprule',
    'Tool & Total time & Peak RSS & Datasets \\\\',
    '\\midrule',
]
for _, row in res.iterrows():
    name = RESOURCE_DISPLAY.get(row['variant'], row['variant'])
    s4.append(
        f'{name} & {fmt_time(row["total_wall_s"])}'
        f' & {fmt_mem(row["max_peak_rss_mb"])}'
        f' & {int(row["n_datasets"])} \\\\'
    )
s4 += ['\\bottomrule', '\\end{tabular}']

table_s4_body = '\n'.join(s4)
print(table_s4_body)

\begin{tabular}{lrrr}
\toprule
Tool & Total time & Peak RSS & Datasets \\
\midrule
SenseXAMP & 17.8\,h & 39.5\,GB & 8 \\
DeepAMP & 44.5\,min & 7.8\,GB & 8 \\
AMPlify & 41.4\,min & 15.4\,GB & 8 \\
MBC-Attention & 27.1\,min & 5.4\,GB & 8 \\
AMPEPPy & 17.3\,min & 3.1\,GB & 8 \\
APEX & 15.5\,min & 1.4\,GB & 8 \\
AMPScanner & 15.2\,min & 1.5\,GB & 8 \\
AMPredictor & 10.2\,min & 32.8\,GB & 7 \\
AMPredMFA & 7.8\,min & 7.2\,GB & 8 \\
HydrAMP$_{\mathrm{AMP}}$ & 6.1\,min & 673\,MB & 8 \\
HydrAMP$_{\mathrm{MIC}}$ & 5.2\,min & 627\,MB & 8 \\
\bottomrule
\end{tabular}


## 8. Assembled supplement LaTeX

In [23]:
# ---- Preamble requirements ----

PREAMBLE = r"""% =============================================================
% BATTLE-AMP Supplementary Tables
% Generated by supplementary_tables.ipynb
%
% Required in preamble:
%   \usepackage{booktabs}
%   \usepackage{amsmath}
%   \usepackage{rotating}       % for sidewaystable
%   \usepackage{caption}        % for \ContinuedFloat
%   \usepackage{textcomp}       % for \textmu
% =============================================================
"""

# ---- Captions ----

S2_CAPTION = (
    'Full classification metrics for all 21 model variants '
    'across 27 benchmark tasks. '
    'Metrics: Coverage (Cov.), Matthews Correlation Coefficient '
    '(MCC), False Positive Rate (FPR), AUROC, AUPRC, '
    'partial AUROC at FPR $\\leq 0.1$ and $\\leq 0.01$ '
    '(pAUROC$_{0.1}$, pAUROC$_{0.01}$; McClish-standardized), '
    'Precision@$k$ ($k = \\min(100, n)$), and positive likelihood '
    'ratio (LR$+$ = TPR/FPR; $\\infty$ where FPR $= 0$). '
    'Best value per metric and task shown in \\textbf{bold}. '
    '$n$ denotes task size.'
)

S3_CAPTION = (
    'Full regression metrics for all 14 regressor variants on '
    'strain-level MIC prediction tasks. '
    'Metrics: Coverage (Cov.), $R^2$ on $\\log_2$-transformed MIC '
    '($R^2_{\\log_2}$), Spearman rank correlation ($\\rho_s$), '
    'and Root Mean Squared $\\log_2$ Error (RMSL$^2$E). '
    'Predictions clamped to 0.25--512~\\textmu g/ml. '
    'Best value per metric and task shown in \\textbf{bold}.'
)

S4_CAPTION = (
    'Resource usage summary. Total wall-clock inference time and '
    'peak resident set size (RSS) across all benchmark datasets. '
    'All runs performed on a single compute node.'
)


# ---- Assemble ----

parts = [PREAMBLE]

first_cls = True
for group_name, group_tasks in TASK_GROUPS.items():
    valid = [t for t in group_tasks if t in cls['task'].values]
    if not valid:
        continue

    body = build_cls_subtable(
        cls, valid, CLS_METRICS, MODEL_ORDER,
        TASK_DISPLAY, CLS_METRIC_LABELS, MODEL_DISPLAY,
        group_label=group_name,
    )

    parts.append('\\begin{sidewaystable}[p]')
    parts.append('\\centering')
    parts.append('\\scriptsize')

    if first_cls:
        parts.append(f'\\caption{{{S2_CAPTION}}}')
        parts.append('\\label{tab:s2_classification}')
        first_cls = False
    else:
        parts.append('\\ContinuedFloat')
        parts.append(
            '\\caption[]{Table~\\ref{tab:s2_classification}'
            ' (continued).}')

    parts.append('')
    parts.append(body)
    parts.append('')
    parts.append('\\end{sidewaystable}')
    parts.append('')

# Regression: portrait, footnotesize
parts.append('\\begin{table}[p]')
parts.append('\\centering')
parts.append('\\footnotesize')
parts.append(f'\\caption{{{S3_CAPTION}}}')
parts.append('\\label{tab:s3_regression}')
parts.append('')
parts.append(table_s3_body)
parts.append('')
parts.append('\\end{table}')
parts.append('')

# Resources: portrait, footnotesize
parts.append('\\begin{table}[p]')
parts.append('\\centering')
parts.append('\\footnotesize')
parts.append(f'\\caption{{{S4_CAPTION}}}')
parts.append('\\label{tab:s4_resources}')
parts.append('')
parts.append(table_s4_body)
parts.append('')
parts.append('\\end{table}')

full_tex = '\n'.join(parts)
print(f'Full supplement: {len(full_tex):,} characters')

Full supplement: 64,381 characters


In [24]:
print(full_tex)

% =============================================================
% BATTLE-AMP Supplementary Tables
% Generated by supplementary_tables.ipynb
%
% Required in preamble:
%   \usepackage{booktabs}
%   \usepackage{amsmath}
%   \usepackage{rotating}       % for sidewaystable
%   \usepackage{caption}        % for \ContinuedFloat
%   \usepackage{textcomp}       % for \textmu
% =============================================================

\begin{sidewaystable}[p]
\centering
\scriptsize
\caption{Full classification metrics for all 21 model variants across 27 benchmark tasks. Metrics: Coverage (Cov.), Matthews Correlation Coefficient (MCC), False Positive Rate (FPR), AUROC, AUPRC, partial AUROC at FPR $\leq 0.1$ and $\leq 0.01$ (pAUROC$_{0.1}$, pAUROC$_{0.01}$; McClish-standardized), Precision@$k$ ($k = \min(100, n)$), and positive likelihood ratio (LR$+$ = TPR/FPR; $\infty$ where FPR $= 0$). Best value per metric and task shown in \textbf{bold}. $n$ denotes task size.}
\label{tab:s2_classificati

In [34]:
# ---- Section 8: Assemble full supplement ----

PREAMBLE = r"""% =============================================================
% BATTLE-AMP Supplementary Tables
% Generated by supplementary_tables.ipynb
%
% Required in document preamble:
%   \usepackage{booktabs}
%   \usepackage{amsmath}
%   \usepackage{caption}
%   \usepackage{textcomp}       % \textmu
%
% Before first supplementary table:
%   \renewcommand{\thetable}{S\arabic{table}}
%   \setcounter{table}{1}       % so first table = S2
% =============================================================
"""

S2_FULL_CAPTION = (
    'Classification metrics for the \\textbf{AMP/non-AMP} task. '
    'Metrics reported throughout Tables~S2--S28: '
    'Coverage (Cov.), Matthews Correlation Coefficient '
    '(MCC), False Positive Rate (FPR), AUROC, AUPRC, '
    'partial AUROC at FPR $\\leq 0.1$ and $\\leq 0.01$ '
    '(pAUROC$_{0.1}$, pAUROC$_{0.01}$; McClish-standardized), '
    'Precision@$k$ ($k = \\min(100, n)$), and positive likelihood '
    'ratio (LR$+$ = TPR/FPR; $\\infty$ where FPR $= 0$). '
    'Best value per metric shown in \\textbf{bold}. '
    '$n$ denotes task size.'
)

S3_CAPTION = (
    'Full regression metrics for all 14 regressor variants on '
    'strain-level MIC prediction tasks. '
    'Metrics: Coverage (Cov.), $R^2$ on $\\log_2$-transformed MIC '
    '($R^2_{\\log_2}$), Spearman rank correlation ($\\rho_s$), '
    'and Root Mean Squared $\\log_2$ Error (RMSL$^2$E). '
    'Predictions clamped to 0.25--512~\\textmu g/ml. '
    'Best value per metric and task shown in \\textbf{bold}.'
)

S4_CAPTION = (
    'Resource usage summary. Total wall-clock inference time and '
    'peak resident set size (RSS) across all benchmark datasets. '
    'All runs performed on a single compute node.'
)


# ---- Assemble ----

parts = [PREAMBLE]

parts.append('\\renewcommand{\\thetable}{S\\arabic{table}}')
parts.append('\\setcounter{table}{1}')
parts.append('')

# --- Classification: one table* per task ---
is_first = True
for group_name, group_tasks in TASK_GROUPS.items():
    valid = [t for t in group_tasks if t in cls['task'].values]
    if not valid:
        continue

    for task in valid:
        td = cls[cls['task'] == task]
        task_label = TASK_DISPLAY.get(task, task)
        n_obs = int(td['task_size'].iloc[0])

        if is_first:
            caption = (
                'Classification metrics for the \\textbf{AMP/non-AMP} task. '
                'Metrics reported throughout '
                'Tables~\\ref{tab:s2_classification}--\\ref{tab:cls_slay}: '
                'Coverage (Cov.), Matthews Correlation Coefficient '
                '(MCC), False Positive Rate (FPR), AUROC, AUPRC, '
                'partial AUROC at FPR $\\leq 0.1$ and $\\leq 0.01$ '
                '(pAUROC$_{0.1}$, pAUROC$_{0.01}$; McClish-standardized), '
                'Precision@$k$ ($k = \\min(100, n)$), and positive likelihood '
                'ratio (LR$+$ = TPR/FPR; $\\infty$ where FPR $= 0$). '
                'Best value per metric shown in \\textbf{bold}. '
                '$n$ denotes task size.'
            )
            label = 'tab:s2_classification'
            is_first = False
        else:
            caption = (
                f'Classification metrics for the '
                f'\\textbf{{{group_name}}}: {task_label} task '
                f'($n = {n_obs:,}$). '
                f'Metrics as defined in Table~\\ref{{tab:s2_classification}}.'
            )
            label = f'tab:cls_{task}'

        body = build_cls_subtable(
            cls, [task], CLS_METRICS, MODEL_ORDER,
            TASK_DISPLAY, CLS_METRIC_LABELS, MODEL_DISPLAY,
            group_label=group_name,
        )

        parts.append('\\begin{table*}[!hptb]')
        parts.append('\\centering')
        parts.append('\\footnotesize')
        parts.append(f'\\caption{{{caption}}}')
        parts.append(f'\\label{{{label}}}')
        parts.append('')
        parts.append(body)
        parts.append('')
        parts.append('\\end{table*}')
        parts.append('')

# --- Regression (two-column, footnotesize) ---
parts.append('\\begin{table*}[!hptb]')
parts.append('\\centering')
parts.append('\\footnotesize')
parts.append(f'\\caption{{{S3_CAPTION}}}')
parts.append('\\label{tab:s_regression}')
parts.append('')
parts.append(table_s3_body)
parts.append('')
parts.append('\\end{table*}')
parts.append('')

# --- Resources (single column is fine) ---
parts.append('\\begin{table}[p]')
parts.append('\\centering')
parts.append('\\footnotesize')
parts.append(f'\\caption{{{S4_CAPTION}}}')
parts.append('\\label{tab:s_resources}')
parts.append('')
parts.append(table_s4_body)
parts.append('')
parts.append('\\end{table}')

full_tex = '\n'.join(parts)
print(f'Full supplement: {len(full_tex):,} characters')
print(f'  Total tables: {full_tex.count(chr(92) + "begin{table")}')

Full supplement: 70,037 characters
  Total tables: 29


In [35]:
print(full_tex)

% =============================================================
% BATTLE-AMP Supplementary Tables
% Generated by supplementary_tables.ipynb
%
% Required in document preamble:
%   \usepackage{booktabs}
%   \usepackage{amsmath}
%   \usepackage{caption}
%   \usepackage{textcomp}       % \textmu
%
% Before first supplementary table:
%   \renewcommand{\thetable}{S\arabic{table}}
%   \setcounter{table}{1}       % so first table = S2
% =============================================================

\renewcommand{\thetable}{S\arabic{table}}
\setcounter{table}{1}

\begin{table*}[!hptb]
\centering
\footnotesize
\caption{Classification metrics for the \textbf{AMP/non-AMP} task. Metrics reported throughout Tables~\ref{tab:s2_classification}--\ref{tab:cls_slay}: Coverage (Cov.), Matthews Correlation Coefficient (MCC), False Positive Rate (FPR), AUROC, AUPRC, partial AUROC at FPR $\leq 0.1$ and $\leq 0.01$ (pAUROC$_{0.1}$, pAUROC$_{0.01}$; McClish-standardized), Precision@$k$ ($k = \min(100, n)$), an

In [20]:
# Print first ~120 lines for inspection
for line in full_tex.split('\n')[:120]:
    print(line)

% =============================================================
% BATTLE-AMP Supplementary Tables
% Generated by supplementary_tables.ipynb
%
% Required packages: booktabs, amsmath, caption (for captionsetup),
%                    textcomp (for \textmu)
% =============================================================

\begin{table}[p]
\centering
\footnotesize
\caption{Full classification metrics for all 21 model variants across 27 benchmark tasks. Metrics: Coverage (Cov.), Matthews Correlation Coefficient (MCC), False Positive Rate (FPR), AUROC, AUPRC, partial AUROC at FPR $\leq 0.1$ and $\leq 0.01$ (pAUROC$_{0.1}$, pAUROC$_{0.01}$; McClish-standardized), Precision@$k$ ($k = \min(100, n)$), and positive likelihood ratio (LR$+$ = TPR/FPR; $\infty$ where FPR $= 0$). Best value per metric and task shown in \textbf{bold}. $n$ denotes task size.}
\label{tab:s2_classification}

% --- AMP/non-AMP: AMP/non-AMP ---
\begin{tabular}{lrrrrrrrrr}
\toprule
\multicolumn{10}{l}{\textbf{AMP/non-AMP}: AM

## 9. Sanity checks

In [21]:
# Best model per classification task by MCC
best_mcc = (
    cls.loc[cls.groupby('task')['mcc'].idxmax()]
    [['task', 'variant', 'mcc', 'coverage', 'lr_plus']]
    .sort_values('mcc', ascending=False)
    .reset_index(drop=True)
)
best_mcc['variant'] = best_mcc['variant'].map(
    lambda v: MODEL_DISPLAY.get(v, v))
best_mcc['task'] = best_mcc['task'].map(
    lambda t: TASK_DISPLAY.get(t, t).replace('\\', ''))
best_mcc.columns = ['Task', 'Best model', 'MCC', 'Cov.', 'LR+']
print('Best model per classification task (by MCC):')
display(best_mcc)

Best model per classification task (by MCC):


,Task,Best model,MCC,Cov.,LR+
0,SyntheticRandom,SenseXAMP$_{\mathrm{Ec}}$,0.910248,0.837868,190.829782
1,AMP/non-AMP,HydrAMP$_{\mathrm{AMP}}$,0.805908,0.546254,7.349711
2,1--10 aa,AMPredictor,0.752221,1.000000,3.753214
3,textit{E.~coli},MBC-Attention,0.704532,0.996677,2.940201
4,Gram$-$,MBC-Attention,0.693993,0.996760,2.947329
5,31--50 aa,MBC-Attention,0.686680,1.000000,5.292887
6,40% identity,MBC-Attention,0.683357,0.977124,4.149829
7,60% identity,MBC-Attention,0.676156,0.992063,4.330539
8,80% identity,MBC-Attention,0.651207,0.994786,3.501592
9,21--30 aa,MBC-Attention,0.645715,1.000000,3.011975


In [22]:
# Best model per regression task by Spearman
best_sp = (
    reg.loc[reg.groupby('task')['spearman'].idxmax()]
    [['task', 'variant', 'spearman', 'r2_log2', 'rmsl2e']]
    .sort_values('spearman', ascending=False)
    .reset_index(drop=True)
)
best_sp['variant'] = best_sp['variant'].map(
    lambda v: MODEL_DISPLAY.get(v, v))
best_sp['task'] = best_sp['task'].map(
    lambda t: REG_TASK_DISPLAY.get(t, t).replace('\\', ''))
best_sp.columns = [
    'Task', 'Best model', 'Spearman', 'R2_log2', 'RMSL2E']
print('Best model per regression task (by Spearman):')
display(best_sp)

Best model per regression task (by Spearman):


,Task,Best model,Spearman,R2_log2,RMSL2E
0,textit{S.~aureus} ATCC 25923,DeepAMP-CNN-G$-$,0.532345,0.122203,2.047218
1,textit{E.~coli} ATCC 25922,MBC-Attention,0.531945,0.232534,1.736895


In [23]:
print('Task group summary:')
for group, tasks in TASK_GROUPS.items():
    n = len([t for t in tasks if t in cls['task'].values])
    print(f'  {group:25s}  {n}/{len(tasks)} tasks')
total_cls = sum(
    len([t for t in ts if t in cls['task'].values])
    for ts in TASK_GROUPS.values())
print(f'\n  Total classification tasks: {total_cls}')
print(f'  Regression tasks: {len(valid_reg_tasks)}')

Task group summary:
  AMP/non-AMP                1/1 tasks
  GeneralActivity            1/1 tasks
  GramActivity               2/2 tasks
  SpeciesActivity            5/5 tasks
  StrainActivity             7/7 tasks
  LengthSplit                4/4 tasks
  HomologySplit              3/3 tasks
  Synthetic decoys           3/3 tasks
  SLAY                       1/1 tasks

  Total classification tasks: 27
  Regression tasks: 2
